In [16]:
import os
import cv2
import numpy as np
from glob import glob
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import segmentation_models_pytorch as smp

In [17]:
# --- CONFIG ---
IMG_SIZE = (360, 640)
BATCH_SIZE = 4
EPOCHS = 10
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [18]:
class BDDSegmentationDataset(Dataset):
    def __init__(self, image_dir, mask_dir):
        self.image_dir = image_dir
        self.mask_dir = mask_dir

        # List all jpg and png files
        image_files = sorted(glob(os.path.join(image_dir, "*.jpg")))
        mask_files = sorted(glob(os.path.join(mask_dir, "*.png")))

        # ✅ Match only filenames that exist in both
        self.samples = []
        mask_names = {os.path.basename(m).replace(".png", ""): m for m in mask_files}

        for img_path in image_files:
            base = os.path.basename(img_path).replace(".jpg", "")
            if base in mask_names:
                self.samples.append((img_path, mask_names[base]))

        if len(self.samples) == 0:
            raise RuntimeError(f"No matching image-mask pairs found in:\n{image_dir}\n{mask_dir}")

        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Resize(IMG_SIZE)
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        img = cv2.resize(img, IMG_SIZE[::-1])
        mask = cv2.resize(mask, IMG_SIZE[::-1])
        mask = (mask > 0).astype(np.float32)

        img = self.transform(img)
        mask = torch.tensor(mask).unsqueeze(0)
        return img, mask


In [19]:
# --- MODEL ---
model = smp.Unet(
    encoder_name="resnet34", 
    encoder_weights="imagenet", 
    in_channels=3, 
    classes=1
).to(DEVICE)

In [20]:
# --- TRAINING ---
def train(model, dataloader, epochs):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    model.train()
    for epoch in range(epochs):
        epoch_loss = 0
        for imgs, masks in tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}"):
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        print(f"✅ Epoch {epoch+1} | Loss: {epoch_loss/len(dataloader):.4f}")

    torch.save(model.state_dict(), "lane_segmentation_model.pth")
    print("🎯 Model saved: lane_segmentation_model.pth")

In [22]:
if __name__ == "__main__":
    img_dir = r"C:\Users\patel\Downloads\Lane_detection\bdd100k\images\100k\train"
    mask_dir = r"C:\Users\patel\Downloads\Lane_detection\Lane_detection\processed_lane_masks\train"
    dataset = BDDSegmentationDataset(img_dir, mask_dir)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    train(model, dataloader, EPOCHS)

Epoch 1/10: 100%|████████████████████████████████████████████████████████████████████| 289/289 [29:03<00:00,  6.03s/it]


✅ Epoch 1 | Loss: 0.2196


Epoch 2/10: 100%|████████████████████████████████████████████████████████████████████| 289/289 [27:43<00:00,  5.76s/it]


✅ Epoch 2 | Loss: 0.0689


Epoch 3/10:  40%|███████████████████████████                                         | 115/289 [09:32<14:26,  4.98s/it]

KeyboardInterrupt

